# Garbage Classification System — Colab Training Notebook

This notebook trains a CNN classifier using TrashNet-style folder data, evaluates it on a held-out validation split, and optionally runs YOLO inference / TACO fine-tuning preparation.

**Before running:** upload the TrashNet dataset and set `DATA_DIR` below. Expected folder layout: `cardboard/`, `glass/`, `metal/`, `paper/`, `plastic/`, `trash/`.

A generic YOLO model is not a litter-specific detector. Use custom TACO-trained weights for garbage/litter detection claims.


In [ ]:
# 1. Install dependencies
!pip -q install tensorflow scikit-learn matplotlib ultralytics opencv-python-headless


In [ ]:
# 2. Imports and runtime check
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


In [ ]:
# 3. Set dataset path
# Option A: upload/extract TrashNet into /content/trashnet-resized
# Option B: mount Google Drive and point DATA_DIR to your dataset folder.
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path('/content/trashnet-resized')  # CHANGE THIS to the folder containing six class subfolders
MODEL_DIR = Path('/content/drive/MyDrive/GarbageProject/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found: {DATA_DIR}. Upload/extract TrashNet and update DATA_DIR.")
print("Class folders:", sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()]))


In [ ]:
# 4. Inspect class counts
from collections import Counter
valid_ext = {'.jpg', '.jpeg', '.png', '.bmp'}
counts = {p.name: sum(1 for f in p.rglob('*') if f.suffix.lower() in valid_ext)
          for p in DATA_DIR.iterdir() if p.is_dir()}
print(counts)
plt.figure(figsize=(8,4))
plt.bar(counts.keys(), counts.values())
plt.xticks(rotation=30)
plt.ylabel("Image count")
plt.title("Dataset class distribution")
plt.tight_layout()
plt.show()


In [ ]:
# 5. Load train/validation split
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='training', seed=SEED,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, label_mode='int'
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='validation', seed=SEED,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, label_mode='int'
)
class_names = train_ds.class_names
print("Class order:", class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(500).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)


In [ ]:
# 6. Preview a batch
plt.figure(figsize=(10,8))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3,3,i+1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[int(labels[i])])
        plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# 7. OpenCV preprocessing demonstration
import cv2
sample_path = next(p for p in DATA_DIR.rglob('*') if p.suffix.lower() in valid_ext)
bgr = cv2.imread(str(sample_path))
resized = cv2.resize(bgr, (224,224))
denoised = cv2.GaussianBlur(resized, (3,3), 0)
rgb = cv2.cvtColor(denoised, cv2.COLOR_BGR2RGB)
normalized = rgb.astype('float32') / 255.0
plt.figure(figsize=(10,3))
plt.subplot(1,3,1); plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)); plt.title('Original'); plt.axis('off')
plt.subplot(1,3,2); plt.imshow(rgb); plt.title('Resized + denoised'); plt.axis('off')
plt.subplot(1,3,3); plt.imshow(normalized); plt.title('Normalized'); plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# 8. Handcrafted feature extraction: HSV histogram, Canny edge density, GLCM texture
!pip -q install scikit-image
from skimage.feature import graycomatrix, graycoprops

def extract_visual_features(bgr_image):
    image = cv2.resize(bgr_image, (224,224))
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    feats = []
    for channel, bins, ranges in [(0,16,[0,180]),(1,16,[0,256]),(2,16,[0,256])]:
        hist = cv2.calcHist([hsv],[channel],None,[bins],ranges).flatten()
        feats.extend((hist/(hist.sum()+1e-8)).tolist())
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray,100,200)
    edge_density = float(np.mean(edges > 0))
    q = (gray // 32).astype(np.uint8)
    glcm = graycomatrix(q, distances=[1,3], angles=[0,np.pi/4], levels=8, symmetric=True, normed=True)
    for prop in ['contrast','homogeneity','energy','correlation']:
        feats.extend(graycoprops(glcm, prop).flatten().tolist())
    return np.asarray(feats, dtype=np.float32), edge_density

feature_vector, edge_density = extract_visual_features(bgr)
print("Feature vector length:", len(feature_vector), "(48 color + 1 edge + 16 texture)")
print("Edge density:", edge_density)


In [ ]:
# 9. Build and train a compact CNN
augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10)
], name='augmentation')

inputs = keras.Input(shape=(224,224,3))
x = layers.Rescaling(1./255)(inputs)
x = augmentation(x)
x = layers.Conv2D(32,3,padding='same',activation='relu')(x)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(64,3,padding='same',activation='relu')(x)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(128,3,padding='same',activation='relu')(x)
x = layers.MaxPooling2D()(x)
x = layers.Dropout(0.30)(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128,activation='relu')(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(len(class_names),activation='softmax')(x)
model = keras.Model(inputs,outputs)
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(str(MODEL_DIR/'garbage_cnn.keras'), monitor='val_loss', save_best_only=True)
]
history = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)
model.save(MODEL_DIR/'garbage_cnn.keras')
(MODEL_DIR/'class_names.txt').write_text('\n'.join(class_names))


In [ ]:
# 10. Plot training curves
for metric in ['accuracy','loss']:
    plt.figure(figsize=(6,4))
    plt.plot(history.history[metric], label='train')
    plt.plot(history.history['val_'+metric], label='validation')
    plt.xlabel('Epoch'); plt.ylabel(metric); plt.legend(); plt.title(metric.title())
    plt.tight_layout(); plt.show()


In [ ]:
# 11. Evaluate on the held-out validation split
y_true = np.concatenate([y.numpy() for _,y in val_ds])
probabilities = model.predict(val_ds, verbose=1)
y_pred = np.argmax(probabilities, axis=1)
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(xticks_rotation=45, cmap='Blues')
plt.tight_layout(); plt.show()


In [ ]:
# 12. Save evaluation summary
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
acc = accuracy_score(y_true,y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true,y_pred,average='weighted',zero_division=0)
print({'accuracy':float(acc),'weighted_precision':float(precision),
       'weighted_recall':float(recall),'weighted_f1':float(f1)})


In [ ]:
# 13. Optional MobileNetV2 transfer-learning experiment
# This is optional if time/Colab quota is limited.
base = keras.applications.MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')
base.trainable = False
inp = keras.Input(shape=(224,224,3))
x = keras.applications.mobilenet_v2.preprocess_input(inp)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
out = layers.Dense(len(class_names), activation='softmax')(x)
mobile_model = keras.Model(inp,out)
mobile_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                     loss='sparse_categorical_crossentropy', metrics=['accuracy'])
mobile_history = mobile_model.fit(train_ds, validation_data=val_ds, epochs=5)
mobile_model.save(MODEL_DIR/'mobilenet_garbage.keras')


In [ ]:
# 14. Optional YOLO inference
# Generic pretrained COCO weights are only a software demonstration, NOT a litter-specific detector.
!pip -q install ultralytics
from ultralytics import YOLO
yolo = YOLO('yolov8n.pt')
detection_result = yolo.predict(source=str(sample_path), conf=0.25, save=True, verbose=False)
display(Image.open(detection_result[0].save_dir / Path(sample_path).name) if False else detection_result[0].plot())
print("For genuine garbage/litter detection, fine-tune YOLO on TACO annotations and report validation mAP.")


## Optional TACO fine-tuning

The TACO dataset uses COCO-style annotations and a long-tail set of litter categories. A reliable detector requires:
- Downloading the TACO images and its COCO annotation JSON.
- Reviewing category names and deciding a documented mapping.
- Converting the selected COCO boxes into YOLO labels.
- Splitting images into train/validation sets.
- Fine-tuning a YOLO model and evaluating it on held-out annotated images.

Do not label generic COCO YOLO output as TACO-trained garbage detection. If time is short, clearly report the detector as a pretrained demonstration and state that TACO fine-tuning remains future work.


In [ ]:
# 15. Download the trained CNN to your computer
from google.colab import files
files.download(str(MODEL_DIR/'garbage_cnn.keras'))
files.download(str(MODEL_DIR/'class_names.txt'))
